In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
wordsforthewise_lending_club_path = kagglehub.dataset_download('wordsforthewise/lending-club')

print('Data source import complete.')


# Lending Club Loan Default Prediction

This is your original pipeline, cleaned up and made deployment-ready. The
cell order, variable names, and logic are kept as close to the original as
possible. Two things were added, and nothing else was restructured:

1. **One real bug fix:** in the original notebook, the imputation modes,
   category encoders, and the `addr_state` frequency map were all computed
   on the *full* dataset before the train/val/test split. That means
   validation/test (and, in production, brand-new applications) statistics
   leaked into the encoding — and there was no stored, reusable way to
   re-apply the exact same encoding to a new loan application later. Below,
   the train/val/test split now happens *before* those statistics are
   learned, and they're wrapped in a small `fit`/`transform` class
   (`CategoricalEncoder`) so the exact same encoding can be replayed on new
   data at inference time. Everything else that doesn't depend on the data
   (fixed category orders, leakage-column drops, date features, etc.) is
   left exactly where it was.
2. **Save/load** for the fitted encoder + model, and a small inference
   function, so the pipeline can actually be deployed.

Unused/exploratory cells (correlation checks, `.isna()` inspections, index
overlap checks, the Kaggle boilerplate) have been removed.


In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    f1_score, roc_auc_score, accuracy_score, precision_score,
    classification_report, average_precision_score,
)
from xgboost import XGBClassifier


In [ ]:
pd.set_option('display.max_columns', None)

## Load data

In [ ]:
DATA_PATH = "/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz"

df = pd.read_csv(DATA_PATH)


KeyboardInterrupt: 

In [ ]:
df.head()

## Drop columns that are mostly missing

In [ ]:
missing_pct = df.isnull().mean() * 100

col_to_drop = missing_pct[missing_pct >= 80].index

df.drop(columns=col_to_drop, inplace=True)


## Keep only resolved loans (fully paid / charged off) and encode the target

In [ ]:
good = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid",
]
bad = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off"
]

df_filtered = df[df["loan_status"].isin(good + bad)].copy()


In [ ]:
mapping = {
    "Fully Paid": 0,
    "Does not meet the credit policy. Status:Fully Paid": 0,
    "Charged Off": 1,
    "Default": 1,
    "Does not meet the credit policy. Status:Charged Off": 1
}

df_filtered["loan_status"] = df_filtered["loan_status"].map(mapping)


In [ ]:
df_filtered.drop("url", axis=1, inplace=True)

## Drop columns that only exist after the loan outcome is known (data leakage)

In [ ]:
investigate = [
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "last_fico_range_high",
    "last_fico_range_low",
    "hardship_flag",
    "debt_settlement_flag",
]


In [ ]:
df_filtered.drop(columns=investigate, inplace=True)

## Drop identifier / unusable high-cardinality columns

In [ ]:
df_filtered.drop("id", axis=1, inplace=True)
df_filtered.drop("emp_title", axis=1, inplace=True)


In [ ]:
X = df_filtered.drop(columns=["loan_status"])
y = df_filtered["loan_status"]


## Targeted imputation for "months since" columns (no event yet = -1)

In [ ]:
col = ["mths_since_recent_bc_dlq",
       "mths_since_recent_revol_delinq",
       "mths_since_last_major_derog"]

X[col] = X[col].fillna(-1)


## Combine FICO range into a single average feature

In [ ]:
fico_avg = (X["fico_range_low"] + X["fico_range_high"]) / 2

X["fico_avg"] = fico_avg
X.drop(columns=["fico_range_low", "fico_range_high"], inplace=True)


## Define column groups for encoding

In [ ]:
ordinal_cols = [
    "grade",
    "sub_grade",
    "emp_length"
]
nominal_cols = [
    "home_ownership",
    "verification_status",
    "purpose",
    "addr_state"
]
drop_cols = [
    "title",
    "zip_code"
]
binary_cols = [
    "pymnt_plan",
    "initial_list_status",
    "application_type",
    "disbursement_method",
    "term"
]


In [ ]:
X["term"] = X["term"].str.strip()
X["term"] = X["term"].map({
    "36 months": 0,
    "60 months": 1
})


## Drop high-cardinality columns

In [ ]:
X.drop(columns=["title", "zip_code"], inplace=True)

## Date features

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

grade_order = ["A","B","C","D","E","F","G"]

subgrade_order = [
    "A1","A2","A3","A4","A5",
    "B1","B2","B3","B4","B5",
    "C1","C2","C3","C4","C5",
    "D1","D2","D3","D4","D5",
    "E1","E2","E3","E4","E5",
    "F1","F2","F3","F4","F5",
    "G1","G2","G3","G4","G5"
]

emp_order = [
    "< 1 year","1 year","2 years","3 years","4 years",
    "5 years","6 years","7 years","8 years","9 years","10+ years"
]

verification_map = {
    "Not Verified":0,
    "Source Verified":1,
    "Verified":2
}

label_cols = ["home_ownership","purpose"]

# -----------------------------
# Learn everything from train
# -----------------------------

fill_values = {}

for col in ordinal_cols + nominal_cols + binary_cols:
    fill_values[col] = X_train[col].mode(dropna=True)[0]

ordinal_encoder = OrdinalEncoder(
    categories=[grade_order, subgrade_order, emp_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

ordinal_encoder.fit(
    X_train[["grade","sub_grade","emp_length"]]
)

label_maps = {}

for col in label_cols:
    label_maps[col] = {
        value:i
        for i,value in enumerate(sorted(X_train[col].dropna().unique()))
    }

freq_maps = {
    "addr_state":
    X_train["addr_state"].value_counts(normalize=True).to_dict()
}


In [ ]:
def preprocess(df):

    df = df.copy()

    df = df.drop(columns=drop_cols, errors="ignore")

    # Fill Missing
    for col,value in fill_values.items():
        if col in df.columns:
            df[col] = df[col].fillna(value)

    # Ordinal Encoding
    df[["grade","sub_grade","emp_length"]] = ordinal_encoder.transform(
        df[["grade","sub_grade","emp_length"]]
    )

    # Binary Encoding
    df["pymnt_plan"] = df["pymnt_plan"].map({"n":0,"y":1})

    df["initial_list_status"] = df["initial_list_status"].map({
        "f":0,
        "w":1
    })

    df["application_type"] = df["application_type"].map({
        "Individual":0,
        "Joint App":1
    })

    df["disbursement_method"] = df["disbursement_method"].map({
        "Cash":0,
        "DirectPay":1
    })

    df["term"] = (
        df["term"]
        .astype(str)
        .str.extract(r"(\d+)")
        .astype(float)
    )

    df["verification_status"] = df["verification_status"].map(
        verification_map
    )

    # Label Encoding
    for col,mapping in label_maps.items():
        df[col] = df[col].map(mapping).fillna(-1)

    # Frequency Encoding
    df["addr_state"] = (
        df["addr_state"]
        .map(freq_maps["addr_state"])
        .fillna(0)
    )

    df["issue_d"] = pd.to_datetime(
        df["issue_d"],
        format="%b-%Y"
    )

    df["issue_year"] = df["issue_d"].dt.year
    df["issue_month"] = df["issue_d"].dt.month

    df["earliest_cr_line"] = pd.to_datetime(
        df["earliest_cr_line"],
        format="%b-%Y"
    )

    df["credit_history_years"] = (
        (df["issue_d"] - df["earliest_cr_line"]).dt.days / 365.25
    )

    df.drop(
        columns=["issue_d", "earliest_cr_line"],
        inplace=True
    )

    return df

In [ ]:
y.fillna(y.mode()[0], inplace=True)

## Train / validation / test split

Everything up to this point (dropping columns, date features, the FICO
average, binary term mapping) is deterministic — it doesn't depend on any
statistic learned from the data, so it's safe to apply before splitting.

What comes next (mode-imputing remaining missing categoricals, ordinal
encoding `grade`/`sub_grade`/`emp_length`, encoding `home_ownership` /
`purpose`, frequency-encoding `addr_state`) **does** depend on the data
(modes, category codes, frequencies), so the split happens now, and those
steps are fit only on `X_train` below.

In [ ]:
X_train = preprocess(X_train)
X_val = preprocess(X_val)
X_test = preprocess(X_test)

In [ ]:
# Add target back temporarily so each dataset is self-contained
train_processed = X_train.copy()
train_processed["loan_status"] = y_train.values

val_processed = X_val.copy()
val_processed["loan_status"] = y_val.values

test_processed = X_test.copy()
test_processed["loan_status"] = y_test.values


# Save
train_processed.to_csv("processed_train.csv", index=False)
val_processed.to_csv("processed_val.csv", index=False)
test_processed.to_csv("processed_test.csv", index=False)

print("Processed datasets saved successfully.")

print("\nTrain:", train_processed.shape)
print("Validation:", val_processed.shape)
print("Test:", test_processed.shape)

## Train the XGBoost model

In [ ]:
# Compute class imbalance ratio
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

print("Negative:", neg)
print("Positive:", pos)
print("Scale pos weight:", scale_pos_weight)


model2 = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,

    tree_method="hist",

    max_depth=6,
    min_child_weight=5,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    n_estimators=10000,
    early_stopping_rounds=50,

    random_state=42
)


model2.fit(
    X_train,
    y_train,
    eval_set=[
        (X_train, y_train),
        (X_val, y_val)
    ],
    verbose=50
)


# Test probabilities
proba2 = model2.predict_proba(X_test)[:, 1]

print("Best iteration:", model2.best_iteration)
print("Best validation score:", model2.best_score)

Negative: 161811
Positive: 40404
Scale pos weight: 4.004826254826255
[0]	validation_0-aucpr:0.36426	validation_1-aucpr:0.35526
[50]	validation_0-aucpr:0.41392	validation_1-aucpr:0.39197
[100]	validation_0-aucpr:0.43793	validation_1-aucpr:0.40115
[150]	validation_0-aucpr:0.45448	validation_1-aucpr:0.40492
[200]	validation_0-aucpr:0.46947	validation_1-aucpr:0.40687
[250]	validation_0-aucpr:0.48284	validation_1-aucpr:0.40797
[300]	validation_0-aucpr:0.49468	validation_1-aucpr:0.40846
[350]	validation_0-aucpr:0.50687	validation_1-aucpr:0.40895
[400]	validation_0-aucpr:0.51794	validation_1-aucpr:0.40912
[450]	validation_0-aucpr:0.52876	validation_1-aucpr:0.40923
[500]	validation_0-aucpr:0.54019	validation_1-aucpr:0.40938
[550]	validation_0-aucpr:0.54971	validation_1-aucpr:0.40929
[557]	validation_0-aucpr:0.55161	validation_1-aucpr:0.40929
Best iteration: 507
Best validation score: 0.4094104599099652


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

proba2 = model2.predict_proba(X_test)[:, 1]

print("Test ROC-AUC:", roc_auc_score(y_test, proba2))
print("Test PR-AUC:", average_precision_score(y_test, proba2))

Test ROC-AUC: 0.7317347185874357
Test PR-AUC: 0.4087693725103334


In [ ]:
importance = pd.Series(
    model2.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance.head(30))

int_rate                0.108213
term                    0.068000
issue_year              0.029339
home_ownership          0.024328
fico_avg                0.015973
num_tl_120dpd_2m        0.015786
mort_acc                0.015400
acc_open_past_24mths    0.014929
num_actv_rev_tl         0.014830
dti                     0.013935
funded_amnt_inv         0.013680
avg_cur_bal             0.013023
tot_hi_cred_lim         0.012971
funded_amnt             0.012390
annual_inc              0.012302
issue_month             0.012260
verification_status     0.012235
total_cu_tl             0.011763
num_rev_tl_bal_gt_0     0.011433
all_util                0.011410
inq_last_6mths          0.011398
open_rv_24m             0.011273
application_type        0.010917
loan_amnt               0.010703
max_bal_bc              0.010607
installment             0.010590
num_tl_30dpd            0.010348
num_actv_bc_tl          0.010060
purpose                 0.009796
total_bc_limit          0.009710
dtype: flo

In [ ]:
top15_features = importance.head(15).index.tolist()

print("Top 15 features:")
for i, feature in enumerate(top15_features, 1):
    print(i, feature)

Top 15 features:
1 int_rate
2 term
3 issue_year
4 home_ownership
5 fico_avg
6 num_tl_120dpd_2m
7 mort_acc
8 acc_open_past_24mths
9 num_actv_rev_tl
10 dti
11 funded_amnt_inv
12 avg_cur_bal
13 tot_hi_cred_lim
14 funded_amnt
15 annual_inc


In [ ]:
X_train_15 = X_train[top15_features]
X_val_15   = X_val[top15_features]
X_test_15  = X_test[top15_features]

print(X_train_15.shape)

(202215, 15)


In [ ]:
model_15 = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,

    tree_method="hist",

    max_depth=6,
    min_child_weight=5,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    n_estimators=10000,
    early_stopping_rounds=50,

    random_state=42
)

model_15.fit(
    X_train_15,
    y_train,
    eval_set=[
        (X_train_15, y_train),
        (X_val_15, y_val)
    ],
    verbose=50
)

[0]	validation_0-aucpr:0.34777	validation_1-aucpr:0.33763
[50]	validation_0-aucpr:0.40442	validation_1-aucpr:0.38829
[100]	validation_0-aucpr:0.41775	validation_1-aucpr:0.39290
[150]	validation_0-aucpr:0.42594	validation_1-aucpr:0.39399
[200]	validation_0-aucpr:0.43296	validation_1-aucpr:0.39421
[250]	validation_0-aucpr:0.43935	validation_1-aucpr:0.39432
[277]	validation_0-aucpr:0.44278	validation_1-aucpr:0.39421


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=50,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None,
              n_estimators=10000, n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
proba_15 = model_15.predict_proba(X_test_15)[:, 1]

print("Best iteration:", model_15.best_iteration)
print("Best validation PR-AUC:", model_15.best_score)

print("Test ROC-AUC:", roc_auc_score(y_test, proba_15))
print("Test PR-AUC:", average_precision_score(y_test, proba_15))

Best iteration: 227
Best validation PR-AUC: 0.39442561649052527
Test ROC-AUC: 0.7218402900657197
Test PR-AUC: 0.3943567099675389


In [ ]:
for threshold in [0.4, 0.5, 0.6, 0.7, 0.8]:
    pred = (proba_15 >= threshold).astype(int)

    print(f"\n{'='*50}")
    print(f"THRESHOLD: {threshold}")
    print(f"{'='*50}")

    print(classification_report(y_test, pred, digits=4))


THRESHOLD: 0.4
              precision    recall  f1-score   support

           0     0.9126    0.4779    0.6273    161811
           1     0.2809    0.8166    0.4180     40404

    accuracy                         0.5456    202215
   macro avg     0.5967    0.6473    0.5227    202215
weighted avg     0.7863    0.5456    0.5855    202215


THRESHOLD: 0.5
              precision    recall  f1-score   support

           0     0.8871    0.6603    0.7571    161811
           1     0.3278    0.6635    0.4388     40404

    accuracy                         0.6609    202215
   macro avg     0.6075    0.6619    0.5979    202215
weighted avg     0.7754    0.6609    0.6935    202215


THRESHOLD: 0.6
              precision    recall  f1-score   support

           0     0.8598    0.8148    0.8367    161811
           1     0.3868    0.4678    0.4234     40404

    accuracy                         0.7455    202215
   macro avg     0.6233    0.6413    0.6301    202215
weighted avg     0.7653   

In [ ]:
from sklearn.metrics import classification_report

for threshold in [0.4, 0.5, 0.6, 0.7, 0.8]:
    pred = (proba2 >= threshold).astype(int)

    print(f"\n{'='*50}")
    print(f"THRESHOLD: {threshold}")
    print(f"{'='*50}")

    print(classification_report(
        y_test,
        pred,
        digits=4
    ))


THRESHOLD: 0.4
              precision    recall  f1-score   support

           0     0.9105    0.5285    0.6688    161811
           1     0.2955    0.7921    0.4304     40404

    accuracy                         0.5812    202215
   macro avg     0.6030    0.6603    0.5496    202215
weighted avg     0.7877    0.5812    0.6212    202215


THRESHOLD: 0.5
              precision    recall  f1-score   support

           0     0.8860    0.6942    0.7785    161811
           1     0.3440    0.6422    0.4480     40404

    accuracy                         0.6838    202215
   macro avg     0.6150    0.6682    0.6132    202215
weighted avg     0.7777    0.6838    0.7124    202215


THRESHOLD: 0.6
              precision    recall  f1-score   support

           0     0.8600    0.8303    0.8449    161811
           1     0.4029    0.4586    0.4289     40404

    accuracy                         0.7560    202215
   macro avg     0.6314    0.6444    0.6369    202215
weighted avg     0.7686   

In [ ]:
drop_cols = ["loan_status", "grade", "sub_grade"]

X_train = train.drop(columns=drop_cols)
X_val   = val.drop(columns=drop_cols)
X_test  = test.drop(columns=drop_cols)

y_train = train["loan_status"]
y_val   = val["loan_status"]
y_test  = test["loan_status"]

print("Target in X_train?", "loan_status" in X_train.columns)
print("Target in X_val?", "loan_status" in X_val.columns)
print("Target in X_test?", "loan_status" in X_test.columns)

Target in X_train? False
Target in X_val? False
Target in X_test? False


In [ ]:
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

print("Grade present:", "grade" in X_train.columns)
print("Sub-grade present:", "sub_grade" in X_train.columns)
print("Loan status present:", "loan_status" in X_train.columns)

Train: (202215, 87)
Val: (943669, 87)
Test: (202215, 87)
Grade present: False
Sub-grade present: False
Loan status present: False


In [ ]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model2.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

print(feature_importance.head(20))

In [ ]:
import matplotlib.pyplot as plt

top = feature_importance.head(20)

plt.figure(figsize=(10, 8))
plt.barh(top["feature"], top["importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 XGBoost Features")
plt.show()

results = []

for n in [15, 18, 20, 22, 24, 25, 27, 30]:

    selected_features = (
        feature_importance
        .head(n)["feature"]
        .tolist()
    )

    X_train_small = X_train[selected_features]
    X_val_small = X_val[selected_features]
    X_test_small = X_test[selected_features]

    model_small = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        max_depth=6,
        min_child_weight=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        n_estimators=10000,
        early_stopping_rounds=50,
        random_state=42
    )

    model_small.fit(
        X_train_small,
        y_train,
        eval_set=[(X_val_small, y_val)],
        verbose=False
    )

    prob = model_small.predict_proba(X_test_small)[:, 1]

    results.append({
        "features": n,
        "ROC_AUC": roc_auc_score(y_test, prob),
        "PR_AUC": average_precision_score(y_test, prob)
    })

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
print("Best iteration:", model2.best_iteration)
print("Best score:", model2.best_score)


In [ ]:
y_prob = model2.predict_proba(X_val)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_val, y_pred))


In [ ]:
X_train.head()

In [ ]:
for threshold in [0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2]:
    y_pred = (y_prob >= threshold).astype(int)
    print(f"\nThreshold: {threshold}")
    print(classification_report(y_val, y_pred))


In [ ]:
results = []

for n in [15, 18, 20, 22, 24, 25, 27, 30]:

    selected_features = (
        feature_importance
        .head(n)["feature"]
        .tolist()
    )

    X_train_small = X_train[selected_features]
    X_val_small = X_val[selected_features]
    X_test_small = X_test[selected_features]

    model_small = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        max_depth=6,
        min_child_weight=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        n_estimators=10000,
        early_stopping_rounds=50,
        random_state=42
    )

    model_small.fit(
        X_train_small,
        y_train,
        eval_set=[(X_val_small, y_val)],
        verbose=False
    )

    # Probability
    prob = model_small.predict_proba(X_test_small)[:, 1]

    # Threshold
    threshold = 0.6
    pred = (prob >= threshold).astype(int)

    # AUC metrics
    roc_auc = roc_auc_score(y_test, prob)
    pr_auc = average_precision_score(y_test, prob)

    print("\n" + "="*70)
    print(f"{n} FEATURES")
    print("="*70)

    print(f"ROC-AUC : {roc_auc:.4f}")
    print(f"PR-AUC  : {pr_auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, pred, digits=4))

    results.append({
        "features": n,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc
    })

In [ ]:
# Select top 15 features
selected_features = (
    feature_importance
    .head(15)["feature"]
    .tolist()
)

print("Selected 15 features:")
print(selected_features)

# Create reduced datasets
X_train_15 = X_train[selected_features]
X_val_15 = X_val[selected_features]
X_test_15 = X_test[selected_features]

# Train final 15-feature model
model_15 = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    max_depth=6,
    min_child_weight=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_estimators=10000,
    early_stopping_rounds=50,
    random_state=42
)

model_15.fit(
    X_train_15,
    y_train,
    eval_set=[(X_val_15, y_val)],
    verbose=False
)

# Get probabilities ONCE
prob_15 = model_15.predict_proba(X_test_15)[:, 1]

# Test different thresholds
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

for threshold in thresholds:

    pred = (prob_15 >= threshold).astype(int)

    print("\n" + "="*70)
    print(f"15 FEATURES | THRESHOLD = {threshold}")
    print("="*70)

    print(classification_report(
        y_test,
        pred,
        digits=4
    ))

In [ ]:
train = pd.read_csv("/content/processed_test.csv")
val = pd.read_csv("/content/processed_train.csv")
test = pd.read_csv("/content/processed_val.csv")

## Classification Threshold Selection

The classification threshold determines the trade-off between identifying more potential loan defaulters and incorrectly flagging non-defaulters. Therefore, there is no single universally optimal threshold; the appropriate threshold depends on the lender's business objective and risk tolerance.

| Threshold | Defaulter Precision | Defaulter Recall | F1-Score | Suitable Use Case                                                                                                                                                                                                  |
| --------- | ------------------: | ---------------: | -------: | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **0.40**  |                0.29 |         **0.81** |     0.43 | **Risk-first strategy** — suitable when missing a potential defaulter is very costly. The model identifies 81% of actual defaulters but produces more false-positive warnings.                                     |
| **0.50**  |                0.34 |             0.67 | **0.45** | **Balanced strategy** — provides the best balance among the tested thresholds between detecting defaulters and limiting false positives.                                                                           |
| **0.60**  |            **0.40** |             0.49 |     0.44 | **Conservative flagging strategy** — suitable when falsely classifying reliable borrowers as high-risk is costly. Predictions of default are more precise, but approximately half of actual defaulters are missed. |

### Business Interpretation

* **Threshold = 0.40:** Prioritizes **recall**. Best suited for high-risk screening where the institution wants to identify as many potential defaulters as possible and can tolerate additional false alarms.
* **Threshold = 0.50:** Provides the best balance of the thresholds tested above and is used as the default operating threshold below.
* **Threshold = 0.60:** Prioritizes **precision**. Best suited when falsely flagging a good borrower is especially costly.

Re-run the threshold scan above whenever the model is retrained — the exact numbers will shift with the data.

In [ ]:
chosen_threshold = 0.5


## Evaluate on the held-out test set

In [ ]:
test_pred = (proba2 >= chosen_threshold).astype(int)

print(classification_report(y_test, test_pred))
print("Test PR-AUC:", average_precision_score(y_test, proba2))
print("Test ROC-AUC:", roc_auc_score(y_test, proba2))


## Save the pipeline for deployment

`cat_encoder` and `model2` are saved together in one file, along with the
list of high-missing columns dropped at the start and the chosen threshold
— everything a production service needs to score a brand-new loan
application.

In [ ]:
joblib.dump({

    "model": model2,

    "threshold": chosen_threshold,

    "col_to_drop": list(col_to_drop),

    "fill_values": fill_values,

    "ordinal_encoder": ordinal_encoder,

    "label_maps": label_maps,

    "freq_maps": freq_maps

}, ARTIFACT_PATH)